## Exercise: Variables and variable selection

#### In this exercise, we apply variance thresholding to select features from a dataset.

### Learning objectives
#### By the end of this train, you should be able to:
##### - Perform dummy variable encoding
##### - Implement variance thresholding in Python
##### - Use a variance threshold to filter out some features in a dataset

### Exercises
#### We are provided with the ***Crop_yield*** dataset that contains various factors that could influence the yield of a particular crop across different regions

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

In [2]:
# load dataset
df = pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/Python/Crop_yield.csv")
df.head(5)

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


In [3]:
df.shape

(1000, 9)

### Exercise 1

#### Our dataset contains several categorical features: ***Region***, ***Soil_Type***, and *** Crop_Variety***. 

#### Use dummy variable encoding to convert these features into a numerical format suitable for model training. Verify the transformation by displaying the first five rows of the modified dataset.

In [4]:
df.dtypes

Region               object
Temperature         float64
Rainfall            float64
Soil_Type            object
Fertilizer_Usage    float64
Pesticide_Usage     float64
Irrigation            int64
Crop_Variety         object
Yield               float64
dtype: object

In [5]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['float64']).columns.tolist()
# categorical_cols
# numerical_cols

In [6]:
for col in categorical_cols:
    print(f'{col}: {df[col].unique()}')

Region: ['East' 'West' 'North' 'South']
Soil_Type: ['Clayey' 'Sandy' 'Loamy']
Crop_Variety: ['Variety B' 'Variety A' 'Variety C']


In [7]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [8]:
df_encoded

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety B,Crop_Variety_Variety C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,19.540123,543.669287,99.907202,44.913209,1,29.484676,1,0,0,0,0,0,1
996,26.090378,549.562256,237.636041,42.582608,1,28.190250,1,0,0,0,0,0,1
997,25.104737,756.628061,236.604958,19.356913,1,37.860206,0,0,1,0,0,0,0
998,19.604622,607.894675,86.379325,18.025284,0,32.638678,0,0,1,1,0,0,1


In [9]:
df_encoded.shape

(1000, 13)

In [10]:
df_encoded.columns

Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
       'Irrigation', 'Yield', 'Region_North', 'Region_South', 'Region_West',
       'Soil_Type_Loamy', 'Soil_Type_Sandy', 'Crop_Variety_Variety B',
       'Crop_Variety_Variety C'],
      dtype='object')

In [11]:
df_encoded.columns = df_encoded.columns.str.replace("Variety ", "")
df_encoded.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_B,Crop_Variety_C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,0,0,0,1,0,1,0


### Exercise 2

#### We want to determine which variables from the new dataset we will use for model training.

#### Write a function variance_thresholding that will use variance thresholding to filter out features based on a variance threshold. The function should accept two parameters, which are the DataFrame and the threshold value. It should return two DataFrames, one containing only the features that meet the variance threshold criterion, and one containing the scaled DataFrame.

#### **Hint**: Scaling is crucial as it allows the variance thresholding to be applied uniformly across features. Read up on the ***MinMaxScaler()*** function from the ***sklearn.preprocessing*** package

In [14]:
def variance_thresholding(df, th_val):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)
    df_scaled = pd.DataFrame(
        scaled_data,
        columns=df.columns
    )
    
    selector = VarianceThreshold(threshold=th_val)
    selector.fit(df_scaled)
    selected_columns = df_scaled.columns[selector.get_support()]
    selected_data = selector.transform(df_scaled)
    selected_df = pd.DataFrame(
        selected_data,
        columns=selected_columns
    )
    
    return df_scaled, selected_df

### Exercise 3

#### Using the function we created in **Exercise 2**, apply variance threshold filtering to our encoded dataset, with a threshold of 0.03. Compare the number of features before and after applying the variance threshold

In [17]:
variance_thresholding(df_encoded, 0.03)

(     Temperature  Rainfall  Fertilizer_Usage  Pesticide_Usage  Irrigation  \
 0       0.438382  0.775738          0.618965         0.268771         1.0   
 1       0.328839  0.590605          0.825228         0.982393         0.0   
 2       0.576227  0.127152          0.688819         0.382573         0.0   
 3       0.542678  0.850865          0.551529         0.177395         0.0   
 4       0.320350  0.652757          0.240490         0.351366         1.0   
 ..           ...       ...               ...              ...         ...   
 995     0.333422  0.568323          0.198152         0.872882         1.0   
 996     0.523762  0.573030          0.750740         0.814572         1.0   
 997     0.495120  0.738411          0.746603         0.233477         1.0   
 998     0.335296  0.619619          0.143876         0.200160         0.0   
 999     0.577674  0.568093          0.878561         0.715066         1.0   
 
         Yield  Region_North  Region_South  Region_West  Soil_

### Exercise 4

#### Train two linear regression models:
##### a) Using all the available features in our dummy encoded dataset from Exercise 1
##### b) Using only the features selected through the variance thresholding process in Exercise 3

In [39]:
predictor_cols = [col for col in df_encoded.columns if col != 'Yield']
dependent_col = [col for col in df_encoded.columns if col == 'Yield']

X = df_encoded[predictor_cols]
y = df_encoded[dependent_col]

In [32]:
encoded_model = LinearRegression()

In [31]:
encoded_model.fit(X, y)

LinearRegression()

In [57]:
_, selected_df = variance_thresholding(df_encoded, 0.03)

predictor_cols_vr = [col for col in selected_df.columns if col != 'Yield']


X_vr = selected_df[predictor_cols_vr]
y_vr = df_encoded[dependent_col]

In [58]:
variance_model = LinearRegression()

In [59]:
variance_model.fit(X_vr, y_vr)

LinearRegression()

In [61]:
# Apply dummy variable encoding to the categorical variables
df_encoded = pd.get_dummies(df, columns=["Region", "Soil_Type", "Crop_Variety"], dtype=int)

# Display the first few rows of the modified dataset to confirm the transformation
df_encoded.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_East,Region_North,Region_South,Region_West,Soil_Type_Clayey,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety A,Crop_Variety_Variety B,Crop_Variety_Variety C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,1,0,0,0,1,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,0,1,0,0,1,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,0,1,0,0,0,1,0,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,1,0,0,0,0,1,0,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,1,0,0,0,0,1,0,0,1,0


In [62]:
df_encoded.shape

(1000, 16)